# 🌦️ Pipeline CRISP-DM: Imputación de Datos Hidrometeorológicos
## Estación 25001 | Random Forest Espacio-Temporal

---

| Ítem | Detalle |
|------|---------|
| **Dataset** | Estación 25001 — Serie diaria 1961-2018 |
| **Variables** | PRECIP · EVAP · TMAX · TMIN |
| **Metodología** | CRISP-DM (6 fases) |
| **Modelo** | Random Forest con Feature Engineering Temporal |
| **Librería núcleo** | scikit-learn, pandas, numpy, matplotlib, seaborn |

> **Objetivo:** Construir un pipeline reproducible que, partiendo de datos reales con pérdida controlada, estime los valores faltantes y evalúe la calidad de la imputación contra el *Ground Truth* original.


## 📦 0 · Instalación y Configuración de Dependencias

In [ ]:
# ── Instalación (ejecutar solo una vez) ────────────────────────
# !pip install pandas numpy scikit-learn matplotlib seaborn scipy

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ── Estilo global ──────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})
PALETTE = {"PRECIP": "#2196F3", "EVAP": "#FF9800", "TMAX": "#F44336", "TMIN": "#9C27B0"}
SEED = 42
np.random.seed(SEED)

print("Librerías cargadas correctamente.")


---
## 🔍 Fase 1 — Comprensión de los Datos *(Data Understanding)*

### Marco Teórico

La **Comprensión de los Datos** es la primera fase analítica del ciclo CRISP-DM y determina todas las decisiones posteriores.  
En series hidrometeorológicas, los datos faltantes rara vez son aleatorios: su origen puede ser instrumental (falla del sensor), operativo (ausencia del observador) o ambiental (eventos extremos que dañan equipos).

Bajo la taxonomía de Rubin (1976), los mecanismos de pérdida se clasifican en tres categorías:

| Mecanismo | Descripción | Ejemplo en este dataset |
|-----------|-------------|-------------------------|
| **MCAR** – *Missing Completely At Random* | La ausencia es independiente de cualquier variable observada o no observada | TMAX/TMIN: 6–7 registros dispersos a lo largo de 57 años |
| **MAR** – *Missing At Random* | La ausencia depende de otras variables **observadas** | PRECIP: concentración en junio de la década 1960 (cambio de estación) |
| **MNAR** – *Missing Not At Random* | La ausencia está relacionada con el propio valor no observado | EVAP: ausencias prolongadas cuando la evaporación supera umbrales operativos |

> **Consecuencias si no se tratan correctamente:**  
> — Los estimadores de media/varianza quedan sesgados.  
> — Los modelos ML entrenan sobre una distribución truncada.  
> — Las relaciones termodinámicas (e.g. EVAP ↔ TMAX) se distorsionan.  
> — En series de tiempo, la autocorrelación temporal se rompe en los huecos, afectando modelos ARIMA, LSTM y similares.


In [ ]:
# ── 1.1 Carga del dataset ──────────────────────────────────────
FILE_PATH = "25001_preprocesado.csv" 

df = pd.read_csv(FILE_PATH)
df["FECHA"] = pd.to_datetime(df["FECHA"])
df = df.sort_values("FECHA").reset_index(drop=True)

VARS = ["PRECIP", "EVAP", "TMAX", "TMIN"]

print("=" * 55)
print("RESUMEN GENERAL DEL DATASET")
print("=" * 55)
print(f"  Registros totales : {len(df):,}")
print(f"  Rango temporal    : {df['FECHA'].min().date()} → {df['FECHA'].max().date()}")
print(f"  Duración          : {(df['FECHA'].max()-df['FECHA'].min()).days:,} días")
print(f"  Columnas          : {df.columns.tolist()}")
print()
print(df.head(10).to_string(index=False))


In [ ]:
# ── 1.2 Análisis de valores faltantes ─────────────────────────
print("=" * 55)
print("DIAGNÓSTICO DE VALORES FALTANTES")
print("=" * 55)
print(f"  {'Variable':8s} {'N faltantes':>12s} {'% faltante':>12s}  Mecanismo probable")
print("  " + "-" * 55)

miss_info = {}
for col in VARS:
    n = df[col].isna().sum()
    p = 100 * n / len(df)
    miss_info[col] = (n, p)
    mec = {"PRECIP": "MAR",  "EVAP": "MNAR", "TMAX": "MCAR", "TMIN": "MCAR"}[col]
    print(f"  {col:8s} {n:12,d} {p:12.2f}%  {mec}")

print()
# Patrones de co-ocurrencia
mask = df[VARS].isna()
print("  Co-ocurrencia de faltantes:")
print(f"    Filas con algún NaN   : {mask.any(axis=1).sum():,}")
print(f"    Filas con todo NaN    : {mask.all(axis=1).sum():,}")
print(f"    EVAP ∩ PRECIP faltantes : {(mask['EVAP'] & mask['PRECIP']).sum()}")
print(f"    TMAX ∩ TMIN faltantes   : {(mask['TMAX'] & mask['TMIN']).sum()}")


In [ ]:
# ── 1.3 Estadísticas descriptivas ─────────────────────────────
print("ESTADÍSTICAS DESCRIPTIVAS (datos observados)")
print("-" * 55)
print(df[VARS].describe().round(3).to_string())


In [ ]:
# ── 1.4 Visualización completa del análisis de faltantes ───────
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

# (A) Mapa de calor de faltantes (año × mes)
ax_heat = fig.add_subplot(gs[0, :])
df["YEAR"]  = df["FECHA"].dt.year
df["MONTH"] = df["FECHA"].dt.month
pivot = df.pivot_table(index="YEAR", columns="MONTH",
                       values="EVAP", aggfunc=lambda x: x.isna().mean())
sns.heatmap(pivot, cmap="YlOrRd", linewidths=0.3, ax=ax_heat,
            cbar_kws={"label": "Fracción faltante"})
ax_heat.set_title("(A) Fracción de EVAP faltante por Año × Mes", fontweight="bold")
ax_heat.set_xlabel("Mes"); ax_heat.set_ylabel("Año")

# (B) % faltante por variable (barras)
ax_bar = fig.add_subplot(gs[1, 0])
pcts = [miss_info[v][1] for v in VARS]
bars = ax_bar.barh(VARS, pcts, color=[PALETTE[v] for v in VARS], edgecolor="white")
ax_bar.set_xlabel("% de valores faltantes")
ax_bar.set_title("(B) Porcentaje de faltantes por variable", fontweight="bold")
for bar, p in zip(bars, pcts):
    ax_bar.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f"{p:.2f}%", va="center", fontsize=9)

# (C) Faltantes acumulados por año
ax_cum = fig.add_subplot(gs[1, 1])
for col in VARS:
    yearly = df.groupby("YEAR")[col].apply(lambda x: x.isna().sum())
    ax_cum.plot(yearly.index, yearly.values, marker="o", ms=3,
                label=col, color=PALETTE[col])
ax_cum.set_title("(C) Faltantes anuales por variable", fontweight="bold")
ax_cum.set_xlabel("Año"); ax_cum.set_ylabel("N° faltantes / año")
ax_cum.legend(fontsize=8)

# (D) Distribuciones de variables observadas
ax_dist = fig.add_subplot(gs[2, 0])
for col in ["TMAX", "TMIN"]:
    vals = df[col].dropna()
    ax_dist.hist(vals, bins=40, alpha=0.55, label=col,
                 color=PALETTE[col], edgecolor="none", density=True)
ax_dist.set_title("(D) Distribución TMAX / TMIN (°C)", fontweight="bold")
ax_dist.set_xlabel("Temperatura (°C)"); ax_dist.legend()

# (E) Distribución EVAP y PRECIP
ax_ep = fig.add_subplot(gs[2, 1])
ax_ep.hist(df["EVAP"].dropna(), bins=40, alpha=0.55, label="EVAP",
           color=PALETTE["EVAP"], density=True, edgecolor="none")
ax2 = ax_ep.twinx()
ax2.hist(df["PRECIP"].dropna(), bins=60, alpha=0.35, label="PRECIP",
         color=PALETTE["PRECIP"], density=True, edgecolor="none")
ax_ep.set_title("(E) Distribución EVAP / PRECIP", fontweight="bold")
ax_ep.set_xlabel("mm"); ax_ep.set_ylabel("Densidad EVAP", color=PALETTE["EVAP"])
ax2.set_ylabel("Densidad PRECIP", color=PALETTE["PRECIP"])

fig.suptitle("Fase 1 · Análisis de Valores Faltantes — Estación 25001 (1961–2018)",
             fontsize=13, fontweight="bold", y=1.01)
plt.savefig("fase1_analisis_faltantes.png", bbox_inches="tight")
plt.show()
print("Figura guardada: fase1_analisis_faltantes.png")


In [ ]:
# ── 1.5 Correlación inter-variable (datos completos) ──────────
corr = df[VARS].dropna().corr()
fig, ax = plt.subplots(figsize=(6, 5))
mask_tri = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5,
            cbar_kws={"shrink": 0.8})
ax.set_title("Correlación de Pearson — Variables observadas\n"
             "(evidencia de relaciones termodinámicas)", fontweight="bold")
plt.tight_layout()
plt.savefig("fase1_correlacion.png", bbox_inches="tight")
plt.show()

print("Interpretación física:")
print(f"  EVAP–TMAX  r = {corr.loc['EVAP','TMAX']:.3f}  → relación energética fuerte")
print(f"  TMAX–TMIN  r = {corr.loc['TMAX','TMIN']:.3f}  → amplitud térmica diaria")
print(f"  PRECIP–EVAP r = {corr.loc['PRECIP','EVAP']:.3f} → efecto de humedad/nubosidad")


---
## 🛠️ Fase 2 — Preparación de Datos *(Data Preparation)*

### Marco Teórico

Para construir un sistema de evaluación riguroso necesitamos un **Ground Truth**: un subconjunto de datos *realmente* completos contra el cual medir la calidad de la imputación. La estrategia es:

1. **Seleccionar la década más densa** — maximiza la representatividad y minimiza el sesgo de selección.
2. **Imputar los pocos faltantes preexistentes** con interpolación lineal, creando un conjunto 100% completo.
3. **Inyectar nulos artificialmente** (*Artificially Masked Cross-Validation*, AMCV) a una tasa y patrón realistas.
4. **Cortar cronológicamente** en 70/20/10 — imprescindible en series de tiempo para evitar que el modelo "vea el futuro".

#### ¿Por qué corte cronológico y no aleatorio?

En una serie temporal, la observación $x_t$ está correlacionada con $x_{t-1}$ (autocorrelación). Si se hace *shuffling* aleatorio, el modelo vería datos de 1978 al entrenar y luego predice 1975 — lo que equivale a **fuga de información hacia el pasado**. El corte temporal garantiza que el modelo sólo use información pasada para predecir el presente.

$$\text{Train} = [t_0, t_1) \quad \text{Test} = [t_1, t_2) \quad \text{Val} = [t_2, t_3]$$

donde $t_1 - t_0 \approx 0.70 \cdot T_{total}$ y $t_2 - t_1 \approx 0.20 \cdot T_{total}$.


In [ ]:
# ── 2.1 Selección de la década con mayor completitud ──────────
df["DECADE"] = (df["YEAR"] // 10) * 10

print("Completitud por década:")
print(f"  {'Década':8s} {'Días':>6s} {'Completos':>10s} {'%':>7s}")
print("  " + "-" * 35)
best_decade, best_pct = None, 0
for d, g in df.groupby("DECADE"):
    total    = len(g)
    complete = g[VARS].notna().all(axis=1).sum()
    pct      = 100 * complete / total
    flag     = " ← SELECCIONADA" if pct == df.groupby("DECADE")[VARS].apply(
        lambda x: x.notna().all(axis=1).mean()).max() * 100 else ""
    print(f"  {int(d)}s   {total:6d}  {complete:10d}  {pct:6.1f}%{flag}")
    if pct > best_pct:
        best_pct = pct
        best_decade = int(d)

print(f"Década seleccionada: {best_decade}s ({best_pct:.1f}% de completitud)")


In [ ]:
# ── 2.2 Construcción del Ground Truth ─────────────────────────
mask_decade = (df["YEAR"] >= best_decade) & (df["YEAR"] <= best_decade + 9)
gt = df.loc[mask_decade, ["FECHA"] + VARS].copy().reset_index(drop=True)

print(f"Ground Truth — {best_decade}s")
print(f"  Registros       : {len(gt):,}")
print(f"  Faltantes previos: {gt[VARS].isna().sum().sum()}")

# Imputar los escasos faltantes preexistentes con interpolación lineal
for col in VARS:
    if gt[col].isna().sum() > 0:
        gt[col] = gt[col].interpolate(method="linear", limit_direction="both")

print(f"  Faltantes tras interpolación: {gt[VARS].isna().sum().sum()}")
print(f"  → Dataset Ground Truth: 100% completo")

# Guardar Ground Truth
gt.to_csv("ground_truth_1970s.csv", index=False)
print("  Guardado: ground_truth_1970s.csv")


In [ ]:
# ── 2.3 Inyección de nulos artificiales (AMCV) ────────────────
"""
Estrategia de inyección realista:
  - PRECIP : tasa 5%  (eventos extremos / falla de pluviómetro)
  - EVAP   : tasa 8%  (ausencias instrumentales prolongadas)
  - TMAX   : tasa 3%  (falla de termómetro)
  - TMIN   : tasa 3%  (ídem, correlacionado con TMAX)

Para simular realismo, además de nulos independientes se inyectan
"rachas" (gaps consecutivos) de 3-7 días en EVAP, que es la variable
con el patrón MNAR más pronunciado.
"""

MISS_RATES = {"PRECIP": 0.05, "EVAP": 0.08, "TMAX": 0.03, "TMIN": 0.03}

corrupted = gt.copy()
injected_mask = pd.DataFrame(False, index=gt.index, columns=VARS)

rng = np.random.default_rng(SEED)

for col, rate in MISS_RATES.items():
    n_total  = len(corrupted)
    n_target = int(n_total * rate)

    if col == "EVAP":
        # 50% de los faltantes de EVAP como rachas consecutivas
        n_runs     = n_target // 2
        idxs_run   = set()
        attempts   = 0
        while len(idxs_run) < n_runs and attempts < 10_000:
            start     = rng.integers(0, n_total - 7)
            run_len   = rng.integers(3, 8)
            idxs_run.update(range(start, min(start + run_len, n_total)))
            attempts += 1
        idxs_run = list(idxs_run)[:n_runs]
        # Resto: aleatorio
        remaining = n_target - len(idxs_run)
        pool      = [i for i in range(n_total) if i not in idxs_run]
        idxs_rand = rng.choice(pool, size=remaining, replace=False).tolist()
        idxs_all  = idxs_run + idxs_rand
    else:
        idxs_all = rng.choice(n_total, size=n_target, replace=False).tolist()

    corrupted.loc[idxs_all, col] = np.nan
    injected_mask.loc[idxs_all, col] = True

# Reporte de inyección
print("Nulos inyectados artificialmente:")
print(f"  {'Variable':8s} {'N inyectados':>14s} {'% inyectado':>12s}")
print("  " + "-" * 38)
for col in VARS:
    n = injected_mask[col].sum()
    print(f"  {col:8s} {n:14,d} {100*n/len(corrupted):12.2f}%")

# Exportar CSV corrupto
corrupted.to_csv("datos_corruptos_1970s.csv", index=False)
print("Exportado: datos_corruptos_1970s.csv")


In [ ]:
# ── 2.4 División cronológica 70 / 20 / 10 ─────────────────────
n      = len(corrupted)
n_tr   = int(n * 0.70)
n_te   = int(n * 0.20)
n_va   = n - n_tr - n_te

train_df = corrupted.iloc[:n_tr].copy()
test_df  = corrupted.iloc[n_tr : n_tr + n_te].copy()
val_df   = corrupted.iloc[n_tr + n_te :].copy()

print("División cronológica (sin shuffle):")
print(f"  TRAIN : {len(train_df):5d} registros "
      f"({train_df['FECHA'].iloc[0].date()} → {train_df['FECHA'].iloc[-1].date()}) ~70%")
print(f"  TEST  : {len(test_df):5d} registros "
      f"({test_df['FECHA'].iloc[0].date()} → {test_df['FECHA'].iloc[-1].date()}) ~20%")
print(f"  VAL   : {len(val_df):5d} registros "
      f"({val_df['FECHA'].iloc[0].date()} → {val_df['FECHA'].iloc[-1].date()}) ~10%")
print()

# Visualizar la partición
fig, ax = plt.subplots(figsize=(14, 3))
ax.barh(0, len(train_df), left=0,
        color="#4CAF50", height=0.5, label=f"TRAIN 70% ({len(train_df)} días)")
ax.barh(0, len(test_df),  left=len(train_df),
        color="#FF9800", height=0.5, label=f"TEST  20% ({len(test_df)} días)")
ax.barh(0, len(val_df),   left=n_tr+n_te,
        color="#F44336", height=0.5, label=f"VAL   10% ({len(val_df)} días)")
ax.set_xlim(0, n)
ax.set_yticks([]); ax.set_xlabel("Índice temporal (días)")
ax.set_title("División Cronológica del Dataset — Fase 2", fontweight="bold")
ax.legend(loc="lower right", fontsize=9)

# Añadir etiquetas de fecha
for pos, lbl in [(n_tr/2, "TRAIN\n1970–1976"),
                 (n_tr + n_te/2, "TEST\n1977–1978"),
                 (n_tr + n_te + n_va/2, "VAL\n1979")]:
    ax.text(pos, 0, lbl, ha="center", va="center",
            fontsize=8, color="white", fontweight="bold")

plt.tight_layout()
plt.savefig("fase2_split_temporal.png", bbox_inches="tight")
plt.show()


---
## 🤖 Fase 3 — Modelado *(Modeling)*

### Marco Teórico: Random Forest con Feature Engineering Temporal

#### ¿Por qué Random Forest?

El **Random Forest (RF)** es un ensamble de $B$ árboles de decisión entrenados sobre submuestras bootstrap del conjunto de entrenamiento, con selección aleatoria de features en cada nodo. La predicción final es:

$$\hat{y} = \frac{1}{B} \sum_{b=1}^{B} T_b(\mathbf{x})$$

donde $T_b$ es el $b$-ésimo árbol y $\mathbf{x}$ el vector de features.

Para series hidrometeorológicas, el RF es especialmente adecuado porque:

| Propiedad | Relevancia hidrometeorológica |
|-----------|-------------------------------|
| **No paramétrico** | Captura relaciones no lineales (e.g. EVAP satura a altas TMAX) |
| **Robusto a outliers** | Precipitaciones extremas no distorsionan el modelo |
| **Multi-output implícito** | Permite imputar cada variable usando las demás como predictores |
| **Feature importance** | Revela qué variables físicas son más predictoras |
| **Sin supuesto de normalidad** | PRECIP tiene distribución fuertemente asimétrica (0-inflada) |

#### Comparación con alternativas

| Método | Ventajas | Limitaciones |
|--------|----------|--------------|
| **MICE** (*Multiple Imputation by Chained Equations*) | Genera múltiples imputaciones, cuantifica incertidumbre | Asume linealidad, lento en series largas |
| **LSTM / GRU** | Captura memoria temporal larga, ideal para gaps ≥ 30 días | Requiere miles de registros, difícil de sintonizar |
| **Kriging de Regresión** | Óptimo espacialmente (múltiples estaciones) | Requiere red de estaciones, semivariograma estacionario |
| **Interpolación lineal** | Simple, preserva continuidad | No captura estacionalidad ni relaciones físicas |

#### Feature Engineering Temporal

El modelo no recibe solo los valores de otras variables — recibe un **vector de contexto espacio-temporal** construido con:

- **Componentes de Fourier**: capturan el ciclo anual (estacionalidad)  
  $\sin\left(\frac{2\pi \cdot DOY}{365.25}\right)$, $\cos\left(\frac{2\pi \cdot DOY}{365.25}\right)$
- **Lags temporales** $\{1, 2, 3, 7, 14, 30\}$ días: autocorrelación física
- **Medias móviles** $\{7, 30\}$ días: tendencia local
- **Climatología mensual**: normal climática como ancla estadística


In [ ]:
# ── 3.1 Feature Engineering ────────────────────────────────────
BASE_FEAT = ["YEAR_NUM", "MONTH", "DOY",
             "SIN_DOY", "COS_DOY", "SIN_DOY2", "COS_DOY2"]
LAGS      = [1, 2, 3, 7, 14, 30]
WINDOWS   = [7, 30]

def build_features(df_in: pd.DataFrame) -> pd.DataFrame:
    """
    Construye el espacio de features espacio-temporal para el RF.
    Entrada : DataFrame con columnas [FECHA, PRECIP, EVAP, TMAX, TMIN]
    Salida  : DataFrame enriquecido con todas las features derivadas
    """
    d = df_in.copy()
    d["FECHA"]    = pd.to_datetime(d["FECHA"])
    d["YEAR_NUM"] = d["FECHA"].dt.year
    d["MONTH"]    = d["FECHA"].dt.month
    d["DOY"]      = d["FECHA"].dt.dayofyear

    # Componentes armónicos de Fourier (estacionalidad)
    d["SIN_DOY"]  = np.sin(2 * np.pi * d["DOY"] / 365.25)
    d["COS_DOY"]  = np.cos(2 * np.pi * d["DOY"] / 365.25)
    d["SIN_DOY2"] = np.sin(4 * np.pi * d["DOY"] / 365.25)
    d["COS_DOY2"] = np.cos(4 * np.pi * d["DOY"] / 365.25)

    # Lags y ventanas rodantes por variable
    for col in VARS:
        for lag in LAGS:
            d[f"{col}_lag{lag}"] = d[col].shift(lag)
        for win in WINDOWS:
            d[f"{col}_roll{win}"] = d[col].shift(1).rolling(win, min_periods=1).mean()

    # Climatología mensual (normal climática de entrenamiento)
    for col in VARS:
        clim = d.groupby("MONTH")[col].transform(
            lambda x: x.mean() if x.notna().sum() > 0 else np.nan)
        d[f"{col}_clim"] = clim

    return d

# Aplicar a todos los splits
train_feat = build_features(train_df)
test_feat  = build_features(test_df)
val_feat   = build_features(val_df)
all_feat   = build_features(corrupted)  # para imputación final

n_features = len([c for c in train_feat.columns
                  if c not in ["FECHA"] + VARS])
print(f" Features construidas: {n_features} variables por registro")
print(f"   Lags: {LAGS}")
print(f"   Ventanas: {WINDOWS}")
print(f"   Fourier: sin/cos de 1° y 2° armónico")


In [ ]:
# ── 3.2 Función de imputación por Random Forest ───────────────
"""
Orden de imputación iterativo (de menor a mayor % faltante):
  TMAX(3%) → TMIN(3%) → PRECIP(5%) → EVAP(8%)

Cada variable imputada se usa como predictor para la siguiente
(estrategia de "ecuaciones encadenadas" del MICE, aplicada con RF).
"""

IMPUTE_ORDER = [
    ("TMAX",   ["TMIN",   "EVAP",    "PRECIP"]),
    ("TMIN",   ["TMAX",   "EVAP",    "PRECIP"]),
    ("PRECIP", ["TMAX",   "TMIN",    "EVAP"  ]),
    ("EVAP",   ["TMAX",   "TMIN",    "PRECIP"]),
]

# Hiperparámetros del Random Forest
RF_PARAMS = {
    "n_estimators"    : 300,   # Número de árboles — equilibrio bias-varianza
    "max_depth"       : 12,    # Profundidad máxima — controla sobreajuste
    "min_samples_leaf": 5,     # Regularización — mínimo de obs. por hoja
    "max_features"    : "sqrt",# Features por nodo — reduce correlación entre árboles
    "n_jobs"          : -1,    # Paralelismo completo
    "random_state"    : SEED,
}

print("Hiperparámetros del Random Forest:")
for k, v in RF_PARAMS.items():
    print(f"  {k:20s}: {v}")
print()
print(f"  Orden de imputación : {' → '.join([t for t,_ in IMPUTE_ORDER])}")


In [ ]:
# ── 3.3 Entrenamiento e imputación ────────────────────────────
from copy import deepcopy

def get_feature_cols(target: str, other_vars: list, df_feat: pd.DataFrame) -> list:
    """Selecciona features relevantes para imputar `target`."""
    cols = BASE_FEAT.copy()
    # Lags y rolling del propio target
    cols += [c for c in df_feat.columns if c.startswith(f"{target}_")]
    # Otras variables físicas (valores actuales + climatología)
    for ov in other_vars:
        if ov in df_feat.columns:
            cols.append(ov)
        clim_col = f"{ov}_clim"
        if clim_col in df_feat.columns:
            cols.append(clim_col)
    return [c for c in cols if c in df_feat.columns]


def impute_rf(df_train_feat, df_pred_feat, target, other_vars, params):
    """
    Entrena un RF sobre filas completas del TRAIN y predice faltantes.
    Aplica restricciones físicas post-predicción:
      · PRECIP, EVAP ≥ 0
      · TMIN < TMAX - 0.1 °C
    """
    feat_cols  = get_feature_cols(target, other_vars, df_train_feat)
    mask_train = df_train_feat[target].notna()
    mask_pred  = df_pred_feat[target].isna()

    if mask_pred.sum() == 0:
        return df_pred_feat[target].copy(), None

    X_tr = df_train_feat.loc[mask_train, feat_cols].copy()
    y_tr = df_train_feat.loc[mask_train, target]
    X_pr = df_pred_feat.loc[mask_pred,  feat_cols].copy()

    # Rellenar NaN residuales en features con mediana de entrenamiento
    medians = X_tr.median()
    X_tr = X_tr.fillna(medians)
    X_pr = X_pr.fillna(medians)

    model = RandomForestRegressor(**params)
    model.fit(X_tr, y_tr)
    y_hat = model.predict(X_pr)

    # ── Restricciones físicas ──────────────────────────────────
    if target in ("PRECIP", "EVAP"):
        y_hat = np.clip(y_hat, 0, None)
    if target == "TMIN":
        tmax_ref = df_pred_feat.loc[mask_pred, "TMAX"].values
        valid    = ~np.isnan(tmax_ref)
        y_hat[valid] = np.minimum(y_hat[valid], tmax_ref[valid] - 0.1)

    result = df_pred_feat[target].copy()
    result.loc[mask_pred] = y_hat
    return result, model


# ── Entrenamiento iterativo ────────────────────────────────────
print("Entrenando modelos de imputación...")
print("=" * 55)

trained_models = {}
imp_result     = all_feat.copy()   # Dataset completo a imputar

for target, others in IMPUTE_ORDER:
    # Re-construir features con valores ya imputados
    imp_feat = build_features(imp_result[["FECHA"] + VARS])
    tr_feat  = build_features(train_df[["FECHA"] + VARS])
    # Actualizar train con valores imputados parciales para consistencia
    for col in VARS:
        tr_feat[col] = imp_feat.loc[imp_feat.index[:len(tr_feat)], col].values

    result, model = impute_rf(tr_feat, imp_feat, target, others, RF_PARAMS)
    imp_result[target] = result.values
    trained_models[target] = model

    n_imp = all_feat[target].isna().sum()
    top3  = []
    if model is not None:
        feat_cols = get_feature_cols(target, others, tr_feat)
        top3 = sorted(zip(feat_cols, model.feature_importances_),
                      key=lambda x: -x[1])[:3]
    print(f"  {target:8s}: {n_imp:4d} imputados | top-3 features: "
          f"{[f[0] for f in top3]}")

print()
print("Imputación completada.")
print("Valores faltantes restantes:", imp_result[VARS].isna().sum().to_dict())


In [ ]:
# ── 3.4 Importancia de features por variable ──────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, (target, others) in zip(axes, IMPUTE_ORDER):
    model = trained_models[target]
    if model is None:
        ax.set_visible(False); continue

    feat_cols = get_feature_cols(target, others,
                build_features(train_df[["FECHA"] + VARS]))
    feat_imp  = sorted(zip(feat_cols, model.feature_importances_),
                       key=lambda x: -x[1])[:15]
    names, imps = zip(*feat_imp)

    ax.barh(range(len(names)), imps, color=PALETTE[target], alpha=0.8)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
    ax.invert_yaxis()
    ax.set_title(f"Importancia de Features — {target}", fontweight="bold",
                 color=PALETTE[target])
    ax.set_xlabel("Importancia (Gini)")

fig.suptitle("Fase 3 · Random Forest — Top-15 Features por Variable",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fase3_feature_importance.png", bbox_inches="tight")
plt.show()
print("Guardado: fase3_feature_importance.png")


In [ ]:
# ── 3.5 Visualización: valores reales vs imputados ─────────────
# Tomar un subperiodo del Test para visualizar nítidamente
test_idx_start = n_tr
test_idx_end   = n_tr + n_te

gt_test = gt.iloc[test_idx_start:test_idx_end].reset_index(drop=True)
imp_test = imp_result.iloc[test_idx_start:test_idx_end].reset_index(drop=True)
cor_test = corrupted.iloc[test_idx_start:test_idx_end].reset_index(drop=True)

fig, axes = plt.subplots(4, 1, figsize=(15, 13), sharex=True)
for ax, col in zip(axes, VARS):
    fechas = gt_test["FECHA"]
    ax.plot(fechas, gt_test[col], lw=1.2,
            color=PALETTE[col], label="Ground Truth", alpha=0.9)
    mask_imp = cor_test[col].isna()
    ax.scatter(fechas[mask_imp], imp_test.loc[mask_imp, col],
               color="black", s=18, zorder=5, label="Imputado (RF)")
    ax.set_ylabel(col, fontsize=10, fontweight="bold", color=PALETTE[col])
    ax.legend(fontsize=8, loc="upper right")

axes[0].set_title("Fase 3 · Comparación Ground Truth vs Imputación (Test set)",
                  fontweight="bold", fontsize=12)
axes[-1].set_xlabel("Fecha")
plt.tight_layout()
plt.savefig("fase3_gt_vs_imputado.png", bbox_inches="tight")
plt.show()


---
## 📊 Fase 4 — Evaluación *(Evaluation)*

### Marco Teórico de Métricas

#### MSE — Error Cuadrático Medio

$$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

**Interpretación en contexto hidrometeorológico:**  
Al elevar al cuadrado, penaliza severamente los errores grandes. En temperatura, un MSE de 4 °C² implica un RMSE de 2 °C — prácticamente el umbral de alerta de la OMM. En precipitación, un único evento extremo mal imputado domina el MSE, haciéndolo sensible a outliers.

#### R² — Coeficiente de Determinación

$$R^2 = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$

**Interpretación:** Fracción de la varianza de la variable que el modelo explica. $R^2 = 1$ implica imputación perfecta; $R^2 = 0$ implica que el modelo no mejora la media; $R^2 < 0$ implica que el modelo es peor que la media.

#### MAE — Error Absoluto Medio *(métrica complementaria)*

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

**Por qué complementa al MSE:** El MAE es robusto a outliers y se expresa en las mismas unidades que la variable (°C o mm). Permite diagnosticar si el error típico es aceptable operativamente. La diferencia RMSE − MAE indica la presencia de errores grandes aislados.

#### RMSE — Raíz del Error Cuadrático Medio

$$RMSE = \sqrt{MSE}$$

Recupera las unidades originales. Si RMSE ≫ MAE, hay outliers de imputación. La OMM acepta RMSE ≤ 2 °C para temperatura diaria y RMSE ≤ 20% de la media para precipitación.

#### KS-Test *(preservación de distribución)*

Contrasta si la distribución de los valores imputados es estadísticamente igual a la del Ground Truth. Un p-valor > 0.05 indica que no se puede rechazar la hipótesis de que son la misma distribución.


In [ ]:
# ── 4.1 Preparar arrays de evaluación ────────────────────────
"""
Evaluamos SOLO sobre los nulos inyectados artificialmente,
ya que son los únicos puntos donde conocemos el Ground Truth
y el modelo generó una estimación.
"""

eval_results = {}

for col in VARS:
    mask_imp = injected_mask[col]   # Nulos que fueron inyectados

    y_true = gt[col].values[mask_imp]
    y_pred = imp_result[col].values[mask_imp]

    # Filtrar pares válidos (sin NaN residuales)
    valid   = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true  = y_true[valid]
    y_pred  = y_pred[valid]

    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    ks_stat, ks_p = stats.ks_2samp(y_true, y_pred)

    eval_results[col] = {
        "N_imputados": valid.sum(),
        "MSE" : mse,  "RMSE": rmse,
        "MAE" : mae,  "R²"  : r2,
        "KS_stat": ks_stat, "KS_p": ks_p,
    }

print("=" * 72)
print("MÉTRICAS DE EVALUACIÓN — Ground Truth vs Imputación RF")
print("=" * 72)
print(f"  {'Variable':8s} {'N':>6s} {'MSE':>10s} {'RMSE':>10s} "
      f"{'MAE':>10s} {'R²':>8s}  KS-p")
print("  " + "-" * 65)
for col, m in eval_results.items():
    ks_flag = "Sí" if m["KS_p"] > 0.05 else "No"
    print(f"  {col:8s} {m['N_imputados']:6d} {m['MSE']:10.4f} {m['RMSE']:10.4f} "
          f"{m['MAE']:10.4f} {m['R²']:8.4f}  {ks_flag} p={m['KS_p']:.3f}")


In [ ]:
# ── 4.2 Comparación de distribuciones (KS-test visual) ────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, col in zip(axes, VARS):
    mask_imp = injected_mask[col]
    y_true = gt[col].values[mask_imp]
    y_pred = imp_result[col].values[mask_imp]
    valid  = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt, yp = y_true[valid], y_pred[valid]

    m = eval_results[col]

    # Histogramas superpuestos
    bins = np.linspace(min(yt.min(), yp.min()),
                       max(yt.max(), yp.max()), 30)
    ax.hist(yt, bins=bins, alpha=0.55, density=True,
            color=PALETTE[col], label="Ground Truth", edgecolor="none")
    ax.hist(yp, bins=bins, alpha=0.55, density=True,
            color="black",       label="Imputado RF",  edgecolor="none",
            histtype="step", lw=2)

    # KDE suavizado
    from scipy.stats import gaussian_kde
    for arr, color, ls in [(yt, PALETTE[col], "-"), (yp, "black", "--")]:
        if len(arr) > 5:
            kde = gaussian_kde(arr, bw_method=0.3)
            xs  = np.linspace(arr.min(), arr.max(), 200)
            ax.plot(xs, kde(xs), color=color, lw=1.5, ls=ls)

    ax.set_title(f"{col}  |  RMSE={m['RMSE']:.3f}  R²={m['R²']:.3f}  "
                 f"KS p={m['KS_p']:.3f}", fontweight="bold", fontsize=9)
    ax.legend(fontsize=8); ax.set_xlabel(col)

fig.suptitle("Fase 4 · Preservación de Distribución — Ground Truth vs Imputación",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fase4_distribuciones.png", bbox_inches="tight")
plt.show()
print("Guardado: fase4_distribuciones.png")


In [ ]:
# ── 4.3 Scatter plot GT vs Imputado ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, col in zip(axes, VARS):
    mask_imp = injected_mask[col]
    y_true = gt[col].values[mask_imp]
    y_pred = imp_result[col].values[mask_imp]
    valid  = ~(np.isnan(y_true) | np.isnan(y_pred))
    yt, yp = y_true[valid], y_pred[valid]

    m = eval_results[col]
    lim = [min(yt.min(), yp.min()) - 1, max(yt.max(), yp.max()) + 1]

    ax.scatter(yt, yp, alpha=0.5, s=20, color=PALETTE[col], edgecolors="none")
    ax.plot(lim, lim, "k--", lw=1.5, label="Línea perfecta")

    # Línea de regresión
    if len(yt) > 2:
        z = np.polyfit(yt, yp, 1)
        p = np.poly1d(z)
        ax.plot(sorted(yt), p(sorted(yt)), "r-", lw=1.2, label="Regresión RF")

    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel(f"Ground Truth ({col})")
    ax.set_ylabel(f"Imputado ({col})")
    ax.set_title(f"{col}  —  R²={m['R²']:.4f}  RMSE={m['RMSE']:.4f}",
                 fontweight="bold", color=PALETTE[col])
    ax.legend(fontsize=8)

fig.suptitle("Fase 4 · Scatter Ground Truth vs Imputado",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fase4_scatter.png", bbox_inches="tight")
plt.show()


In [ ]:
# ── 4.4 Dashboard resumen de métricas ─────────────────────────
metrics_df = pd.DataFrame(eval_results).T
metrics_df = metrics_df[["N_imputados","MSE","RMSE","MAE","R²","KS_p"]]
metrics_df = metrics_df.round(4)

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
metric_pairs = [("RMSE", "Menor es mejor"),
                ("MAE",  "Menor es mejor"),
                ("R²",   "Mayor es mejor"),
                ("KS_p", "Mayor es mejor (>0.05)")]

for ax, (metric, desc) in zip(axes, metric_pairs):
    vals  = [eval_results[col][metric] for col in VARS]
    colors = [PALETTE[col] for col in VARS]
    bars  = ax.bar(VARS, vals, color=colors, edgecolor="white", alpha=0.85)
    ax.set_title(f"{metric}\n{desc}", fontweight="bold", fontsize=10)
    ax.set_ylabel(metric)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.002 * max(vals),
                f"{v:.4f}", ha="center", va="bottom", fontsize=8)
    if metric == "KS_p":
        ax.axhline(0.05, color="red", ls="--", lw=1.5, label="α=0.05")
        ax.legend(fontsize=8)

fig.suptitle("Fase 4 · Dashboard de Métricas de Evaluación",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fase4_dashboard.png", bbox_inches="tight")
plt.show()

print("Tabla resumen:")
print(metrics_df.to_string())


In [ ]:
# ── 4.5 Verificación de restricciones termodinámicas ──────────
print("=" * 55)
print("VERIFICACIÓN DE RESTRICCIONES TERMODINÁMICAS")
print("=" * 55)

v1 = (imp_result["TMIN"] >= imp_result["TMAX"]).sum()
v2 = (imp_result["PRECIP"] < 0).sum()
v3 = (imp_result["EVAP"]   < 0).sum()

print(f"  TMIN >= TMAX (violación física) : {v1:4d}  {'Sin violaciones' if v1==0 else '⚠ REVISAR'}")
print(f"  PRECIP < 0  (imposible físico)  : {v2:4d}  {'Sin violaciones' if v2==0 else '⚠ REVISAR'}")
print(f"  EVAP < 0    (imposible físico)  : {v3:4d}  {'Sin violaciones' if v3==0 else '⚠ REVISAR'}")
print()

# Preservación de correlaciones
print("PRESERVACIÓN DE CORRELACIONES INTER-VARIABLE")
print(f"  {'Par':15s} {'Original':>10s} {'Imputado':>10s} {'Δ':>8s}  Estado")
print("  " + "-" * 50)
pairs_corr = [("EVAP","TMAX"), ("TMAX","TMIN"), ("PRECIP","TMIN")]
for a, b in pairs_corr:
    co = gt[[a,b]].corr().iloc[0,1]
    ci = imp_result[[a,b]].corr().iloc[0,1]
    delta = abs(co - ci)
    flag = "" if delta < 0.02 else ""
    print(f"  {a}-{b:8s}    {co:10.4f} {ci:10.4f} {delta:8.4f}  {flag}")


In [ ]:
# ── 4.6 Exportar CSV imputado final ───────────────────────────
out = imp_result[["FECHA"] + VARS].copy()
out.to_csv("dataset_imputado_1970s.csv", index=False)

print("=" * 55)
print("ARCHIVOS GENERADOS")
print("=" * 55)
archivos = {
    "ground_truth_1970s.csv"    : "Ground Truth (100% completo)",
    "datos_corruptos_1970s.csv" : "Dataset con nulos inyectados",
    "dataset_imputado_1970s.csv": "Dataset imputado por Random Forest",
    "fase1_analisis_faltantes.png": "Visualización Fase 1",
    "fase2_split_temporal.png"  : "Visualización Fase 2",
    "fase3_feature_importance.png": "Importancia de features",
    "fase3_gt_vs_imputado.png"  : "GT vs Imputado (serie temporal)",
    "fase4_distribuciones.png"  : "Distribuciones comparadas",
    "fase4_scatter.png"         : "Scatter GT vs Imputado",
    "fase4_dashboard.png"       : "Dashboard de métricas",
}
for fname, desc in archivos.items():
    print(f"{fname:<40s} {desc}")

print()
print("🎉 Pipeline CRISP-DM completado exitosamente.")
